# Module 11 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

## Reinforcement Learning with Value Iteration

These are the same maps from Module 1 but the "physics" of the world have changed. In Module 1, the world was deterministic. When the agent moved "south", it went "south". When it moved "east", it went "east". Now, the agent only succeeds in going where it wants to go *sometimes*. There is a probability distribution over the possible states so that when the agent moves "south", there is a small probability that it will go "east", "north", or "west" instead and have to move from there.

There are a variety of ways to handle this problem. For example, if using A\* search, if the agent finds itself off the solution, you can simply calculate a new solution from where the agent ended up. Although this sounds like a really bad idea, it has actually been shown to work really well in video games that use formal planning algorithms (which we will cover later). When these algorithms were first designed, this was unthinkable. Thank you, Moore's Law!

Another approach is to use Reinforcement Learning which covers problems where there is some kind of general uncertainty in the actions. We're going to model that uncertainty a bit unrealistically here but it'll show you how the algorithm works.

As far as RL is concerned, there are a variety of options there: model-based and model-free, Value Iteration, Q-Learning and SARSA. You are going to use Value Iteration.

## The World Representation

As before, we're going to simplify the problem by working in a grid world. The symbols that form the grid have a special meaning as they specify the type of the terrain and the cost to enter a grid cell with that type of terrain:

```
token   terrain    cost 
.       plains     1
*       forest     3
^       hills      5
~       swamp      7
x       mountains  impassible
```

When you go from a plains node to a forest node it costs 3. When you go from a forest node to a plains node, it costs 1. You can think of the grid as a big graph. Each grid cell (terrain symbol) is a node and there are edges to the north, south, east and west (except at the edges).

There are quite a few differences between A\* Search and Reinforcement Learning but one of the most salient is that A\* Search returns a plan of N steps that gets us from A to Z, for example, A->C->E->G.... Reinforcement Learning, on the other hand, returns  a *policy* that tells us the best thing to do in **every state.**

For example, the policy might say that the best thing to do in A is go to C. However, we might find ourselves in D instead. But the policy covers this possibility, it might say, D->E. Trying this action might land us in C and the policy will say, C->E, etc. At least with offline learning, everything will be learned in advance (in online learning, you can only learn by doing and so you may act according to a known but suboptimal policy).

Nevertheless, if you were asked for a "best case" plan from (0, 0) to (n-1, n-1), you could (and will) be able to read it off the policy because there is a best action for every state. You will be asked to provide this in your assignment.

We have the same costs as before. Note that we've negated them this time because RL requires negative costs and positive rewards:

In [1]:
costs = { '.': -1, '*': -3, '^': -5, '~': -7}
costs

{'.': -1, '*': -3, '^': -5, '~': -7}

and a list of offsets for `cardinal_moves`. You'll need to work this into your **actions**, A, parameter.

In [2]:
cardinal_moves = [(0,-1), (1,0), (0,1), (-1,0)]

For Value Iteration, we require knowledge of the *transition* function, as a probability distribution.

The transition function, T, for this problem is 0.70 for the desired direction, and 0.10 each for the other possible directions. That is, if the agent selects "north" then 70% of the time, it will go "north" but 10% of the time it will go "east", 10% of the time it will go "west", and 10% of the time it will go "south". If agent is at the edge of the map, it simply bounces back to the current state.

You need to implement `value_iteration()` with the following parameters:

+ world: a `List` of `List`s of terrain (this is S from S, A, T, gamma, R)
+ costs: a `Dict` of costs by terrain (this is part of R)
+ goal: A `Tuple` of (x, y) stating the goal state.
+ reward: The reward for achieving the goal state.
+ actions: a `List` of possible actions, A, as offsets.
+ gamma: the discount rate

you will return a policy: 

`{(x1, y1): action1, (x2, y2): action2, ...}`

Remember...a policy is what to do in any state for all the states. Notice how this is different than A\* search which only returns actions to take from the start to the goal. This also explains why reinforcement learning doesn't take a `start` state.

You should also define a function `pretty_print_policy( cols, rows, policy)` that takes a policy and prints it out as a grid using "^" for up, "<" for left, "v" for down and ">" for right. Use "x" for any mountain or other impassable square. Note that it doesn't need the `world` because the policy has a move for every state. However, you do need to know how big the grid is so you can pull the values out of the `Dict` that is returned.

```
vvvvvvv
vvvvvvv
vvvvvvv
>>>>>>v
^^^>>>v
^^^>>>v
^^^>>>G
```

(Note that that policy is completely made up and only illustrative of the desired output). Please print it out exactly as requested: **NO EXTRA SPACES OR LINES**.

* If everything is otherwise the same, do you think that the path from (0,0) to the goal would be the same for both A\* Search and Q-Learning?
* What do you think if you have a map that looks like:

```
><>>^
>>>>v
>>>>v
>>>>v
>>>>G
```

has this converged? Is this a "correct" policy? What are the problems with this policy as it is?


In [3]:
def read_world(filename):
    result = []
    with open(filename) as f:
        for line in f.readlines():
            if len(line) > 0:
                result.append(list(line.strip()))
    return result

---

<a id="init_rewards"></a>
## init_rewards

`init_rewards` initializes a reward dict with an associated reward for each state in the applicable world, for use in RL value iteration. **Used by**: [value_iteration](#value-iteration)

* **world** list[list[str]]: the input world from which to initialize the rewards.
* **costs** dict: the cost per specific terrain type.
* **goal_pos** tuple: the position of the goal state in the world as a (row, col) tuple.
* **goal_reward** float: the reward for the goal state.

**returns**: dict[tuple, float]: the dict with the reward in each state.

In [4]:
def init_rewards(world: list[list[str]], costs: dict, goal_pos: tuple, goal_reward: float) -> dict[tuple, float]:
    num_rows = len(world)
    num_cols = len(world[0])
    rewards = {}
    for row in range(num_rows):
        for col in range(num_cols):
            if world[row][col] == "x":
                continue
            rewards[(row, col)] = costs[world[row][col]]
    rewards[goal_pos] = goal_reward
    return rewards

In [5]:
small_world = read_world("small.txt")
r = init_rewards(world=small_world, costs=costs, goal_pos=(len(small_world[0])-1, len(small_world)-1), goal_reward=10)
assert (len(small_world[0])-1, len(small_world)-1) in r
assert r[(len(small_world[0])-1, len(small_world)-1)] == 10
assert max(r.values()) == 10
assert min(r.values()) == -3


<a id="init_val_matrix"></a>
## init_val_matrix

`init_val_matrix` initializes the value matrix to zero, but as a dict, using a tuple key for the location, and float for the value. **Used by**: [value_iteration](#value-iteration)

* **world** list[list[str]]: the input world to use for reinforcement learning training.

**returns**: dict[tuple, float]: the value dict.

In [6]:
def init_val_matrix(world: list[list[str]]) -> dict[tuple, float]:
    num_rows = len(world)
    num_cols = len(world[0])
    value = {}
    for row in range(num_rows):
        for col in range(num_cols):
            if world[row][col] == "x":
                continue
            value[(row, col)] = 0.0
    return value

In [7]:
value = init_val_matrix(small_world)
assert [v == 0.0 for _, v in value]
assert (len(small_world), len(small_world[0])) not in value
assert (len(small_world[0]), len(small_world)) not in value
assert (3, 3) not in value # mountain


<a id="find_mountains"></a>
## find_mountains

`find_mountains` compiles a set of mountain locations as tuples, based on the input world, for use in determining what the valid actions are in the value iteration algorithm.
**Used by**: [value_iteration](#value-iteration)

* **world** list[list[str]]: the input world from which to create the mountain location set.

**returns**: set[tuple]: the set of mountain locations as tuples.

In [8]:
def find_mountains(world: list[list[str]]) -> set[tuple]:
    mountains = []
    num_rows = len(world)
    num_cols = len(world[0])
    for row in range(num_rows):
        for col in range(num_cols):
            if world[row][col] == "x":
                mountains.append((row,col))
    return set(mountains)

In [9]:
small_world = read_world("small.txt")
m = find_mountains(small_world)
assert len(m) == 1
assert (3,3) in m
assert (0,0) not in m

<a id="valid_pos"></a>
## valid_pos

`valid_pos` tests whether a successor state is a valid position within the given world. It checks if the successor state is off the map, or on a mountain, which is not allowed. It returns a bool indicating if the position is valid, allowing the main value iteration algorithm to assess the action. **Used by**: [value_iteration](#value-iteration)

* **world** list[list]: the input world from which to assess valid positions.
* **mountains** list[tuple]: a list of mountain locations within the world.
* **pos** tuple: a tuple of the (row,col) coordinates for evaluation.

**returns**: list[tuple]: the list of mountain locations as tuples.

In [10]:
def valid_pos(world: list[list], mountains: list[tuple], pos: tuple) -> bool:
    max_row = len(world)
    max_col = len(world[0])
    row_pos = pos[0]
    col_pos = pos[1]
    if row_pos < 0 or row_pos >= max_row or col_pos < 0 or col_pos >= max_col or (pos in mountains):
        return False
    return True

In [11]:
small_world = read_world("small.txt")
mountains = find_mountains(small_world)
assert valid_pos(small_world, mountains, (-1, 1)) == False
assert valid_pos(small_world, mountains, (0, -1)) == False
assert valid_pos(small_world, mountains, (0, 0)) == True
assert valid_pos(small_world, mountains, (3, 3)) == False
assert valid_pos(small_world, mountains, (7, 7)) == False
assert valid_pos(small_world, mountains, (7, 5)) == False


<a id="convergence"></a>
## convergence

`convergence` tests to see if the value matrix has converged by comparing the max value of the absolute difference of V with V_last. It will return none if the matrix elements don't match (earlier in the code states with mountains are skipped, and these "holes" should end up as identical between V and V_last). **Used by**: [value_iteration](#value-iteration)

* **value1** dict[tuple, float]: the first value matrix for comparison.
* **value2** dict[tuple, float]: the second value matrix for comparison.
* **epsilon** float: the threshold value that controls the bool that's returned.

**returns**: bool | None: returns None if there was an error, True if the algorithm has converged, or False if it hasn't.

In [12]:
def convergence(value1: dict[tuple, float], value2: dict[tuple, float], epsilon: float) -> bool | None:
    keys1 = set(value1.keys())
    keys2 = set(value2.keys())
    diff_vals = []
    if keys1 != keys2:
        return None
    for key in keys1:
        val = abs(value1[key] - value2[key])
        diff_vals.append(val)
    if max(diff_vals) < epsilon:
        return True
    else:
        return False

In [13]:
value1 = {(0,0): 0.5}
value2 = {(1,1): 0.5}
assert convergence(value1, value2, 0.1) == None
value2 = {(0,0): 0.5}
assert convergence(value1, value2, 0.5) == True
value2 = {(0,0): 1.0}
assert convergence(value1, value2, 0.1) == False
value1 = {(0,0): 0.5, (0,1): 1.0}
value2 = {(0,0): 0.5, (0,1): 1.0}
assert convergence(value1, value2, 0.1) == True

<a id="best_action_value"></a>
## best_action_value

`best_action_value` Finds the max of Q summed across the possible actions for stochastic value iteration, and the corresponding argmax of Q. **Used by**: [value_iteration](#value-iteration)

* **curr_pos** tuple: the current position of the agent in the world as a tuple.
* **actions** list: the possible actions the agent can take.
* **world** list[list]: the input world for the agent to use.
* **mountains** list: the mountain locations in the world.
* **rewards** dict: the reward for each world state based on terrain type.
* **gamma** float: the discount value.
* **V_last** dict: the previous values for the value matrix.


**returns**: tuple[float, tuple]: returns None if there was an error, True if the algorithm has converged, or False if it hasn't.

In [14]:
def best_action_value(curr_pos: tuple, actions: list, world: list[list], mountains: list, rewards: dict, gamma: float, V_last: dict) -> tuple[float, tuple]:
    max_q = -1e10
    best_a = (0, 0)
    succeed_prob = 0.7
    fail_prob = 0.1
    for intended_action in actions:
        expected = 0.0
        for actual_action in actions:
            if actual_action == intended_action:
                prob = succeed_prob
            else:
                prob = fail_prob
            successor_pos = (curr_pos[0] + actual_action[0], curr_pos[1] + actual_action[1])
            if not valid_pos(world, mountains, successor_pos):
                successor_pos = curr_pos
            expected += prob * V_last[successor_pos]
        q = rewards[curr_pos] + gamma * expected
        if q > max_q:
            max_q = q
            best_a = intended_action
    return max_q, best_a

In [15]:
world = [[".", "."], [".", "."]]
mountains = []
rewards = {(0, 0): 0.0, (0, 1): 0.0, (1, 0): 0.0, (1, 1): 0.0}
gamma = 0.9
V_last = {(0,0): 0.0, (0,1): 10.0, (1,0): 1.0, (1,1): 0.0}
curr_pos = (0,0)
assert best_action_value(curr_pos,cardinal_moves,world,mountains,rewards,gamma,V_last)[1] == (0,1)
V_last = {(0,0): 0.0, (0,1): 1.0, (1,0): 10.0, (1,1): 0.0}
assert best_action_value(curr_pos,cardinal_moves,world,mountains,rewards,gamma,V_last)[1] == (1,0)
curr_pos = (1,1)
assert best_action_value(curr_pos,cardinal_moves,world,mountains,rewards,gamma,V_last)[1] == (0,-1)


<a id="value_iteration"></a>
## value_iteration

`value_iteration` runs the stochastic value iteration algorithm for reinforcement learning. It uses an input world, a dictionary of costs per terrain type, a goal position, the reward for that goal position, a list of actions, and a discount value to determine the optimal policy. It checks for convergence at the end of each iteration, and also bails after max_iters iterations if it hasn't converged by then, to avoid infinite loops. **Uses**: [init_rewards](#init_rewards), [init_val_matrix](#init_val_matrix), [find_mountains](#find_mountains), [valid_pos](#valid_pos), [convergence](#convergence), [best_action_value](#best_action_value)

* **world** list[list]: the world to run the value iteration algorithm on.
* **costs** dict: the cost lookup per terrain type.
* **goal** tuple: the goal position within the world.
* **reward** float: the reward value for reaching the goal.
* **actions** list: the possible actions that can be taken in each state.
* **gamma** float: the discount value for the algorithm.

**returns**: dict[tuple, tuple]: returns the optimal policy as a dict with the state/position as key and the cardinal move as value.

In [16]:
def value_iteration(world: list[list], costs: dict, goal: tuple, reward: float, actions: list, gamma: float) -> dict[tuple, tuple]:
    epsilon = 0.01
    policy = {}
    rewards = init_rewards(world=world, costs=costs, goal_pos=goal, goal_reward=reward)
    V, mountains = init_val_matrix(world), find_mountains(world)
    iteration = 0
    max_iters = 10000
    for _ in range(max_iters):
        V_last = V.copy()
        for row in range(len(world)):
            for col in range(len(world[0])):
                curr_pos = (row, col)
                if not valid_pos(world, mountains, curr_pos): continue
                V[curr_pos], policy[curr_pos] = best_action_value(curr_pos, actions, world, mountains, rewards, gamma, V_last)
        if convergence(V, V_last, epsilon):
            print(f"Converged after {iteration} iterations.")
            return policy
        iteration += 1
    print(f"Did not converage after {max_iters} iterations.")

<a id="pretty_print_policy"></a>
## pretty_print_policy

`pretty_print_policy` print the policy derived from the stochastic value iteration algorithm as a sequence of string rows and columns.

* **cols**: int: the number of columns in the world.
* **rows** int: the number of rows in the world.
* **policy** dict[tuple, tuple]: the policy derived from the value iteration algorithm, as a dict of states and actions.
* **goal** tuple: the state/position of the goal.

In [17]:
def pretty_print_policy(cols: int, rows: int, policy: dict[tuple, tuple], goal: tuple) -> None:
    actions = {(0,1): ">", (0,-1): "<", (1,0): "v", (-1,0): "^"}
    grid = [["x" for _ in range(cols)] for _ in range(rows)]
    for k,v in policy.items():
        row = k[0]
        col = k[1]
        grid[row][col] = actions[v]
    grid[goal[0]][goal[1]] = "G"
    for row in grid:
        print("".join(row))



## Value Iteration

### Small World

In [18]:
small_world = read_world("small.txt")

In [19]:
num_rows = len(small_world)
num_cols = len(small_world[0])
goal = (num_rows-1, num_cols-1)
gamma = 0.9
small_policy = value_iteration(world=small_world, costs=costs, goal=goal, reward=1000, actions=cardinal_moves, gamma=gamma)

Converged after 107 iterations.


In [20]:
rows = len(small_world)
cols = len(small_world[0])
pretty_print_policy(cols, rows, small_policy, goal)

v>>>vv
vv>>vv
vvv>vv
vvvxvv
>>>>vv
>>>>>v
>>>>>G


### Large World

In [21]:
large_world = read_world( "large.txt")

In [22]:
num_rows = len(large_world)
num_cols = len(large_world[0])
goal = (num_rows-1, num_cols-1)
gamma = 0.9
large_policy = value_iteration(world=large_world, costs=costs, goal=goal, reward=1e8, actions=cardinal_moves, gamma=gamma)

Converged after 216 iterations.


In [23]:
rows = len(large_world)
cols = len(large_world[0])
pretty_print_policy(cols, rows, large_policy, goal)

v>>>>>>>>>>>>>>vv>>>>>>>>vv
vv>>>>>>>>>>>>vvv<xxxxxxxvv
vvvvxx>>>>>>>>>vvxxxvvvxxvv
vvvv<xxx>>>>>>>>>>>vvv<xxvv
vvvv<xxv>>>>>>>>>v>vvvxxxvv
vvv<xxvvv>>>>>>>>v>>vvvxvvv
vvvxxvvvvv>^xxx>>v>>>>>vvvv
v>>>>vvvvvv^<<xxx>>>>>vvvvv
v>>>vvvvvvv<<<xx>>>>>>>vvvv
vv>>vvvvvv<xxxx>>>>>>>>vvvv
v>>>>vvvv<xxx>>>>>vvxxxvvvv
>>>>>vvvvxxv>>>>>>>vvxxvvvv
>>>>>>vvvxxv>>>>>>>>vx>vvvv
>>>>>>>>>>vv>>>>>>>>>>>vvvv
vv>^x>v>>vvv<>>>>>>>>^xvvvv
vv<xxx>>>>vvxxx>>>>>^xxvvvv
vvxx>>>>>>>>>vxxx>^xxxvvvvv
vvvxx>>>>>>>>>>vxxxx>>vvvvv
vvvxxx>>>>>>>>>>>>>>>vvvvvv
vvvvxxx>>>>>>>>>>>>>>vvvvvv
v>>>vvxx>>>>>^x>>>>>>vvvvvv
v>>>>vvxxx>^xx>>>>>>>>vvvvv
>>>>>>>vvxxxx>>>>>>>>>>vvvv
>>>>>>>>>vv>>>>>^xx>>>>vvvv
vx>>>>>>>vvxxx>^xxvxx>>vvvv
vxxx>>>>>>>vxxxx>>>vxxx>>vv
>>>>>>>>>>>>>>>>>>>>>>>>>>G


## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.